In [1]:
!pip install tensorflow pandas scikit-learn keras


[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
import numpy as np
import tensorflow as tf
import pandas as pd
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error

In [2]:
#import the data from csv file
csv_filename= "../datacleaning-pipeline/pnw_model_dataset.csv.gz"
df = pd.read_csv(csv_filename, dtype={'gauge_id':str}, parse_dates=['date'])

#selction of the variables
input_columns = ['prcp_z', 'tmax_z', 'srad_z','vp_z', 'dayl_z', 'drainage_area_km2_z', 'elevation_m_z', 'slope_m_per_km_z','frac_forest_z', 'mean_precip_mm_day_z', 'mean_pet_mm_day_z']
target_column = 'log_runoff'

In [3]:
#Format data for input to LSTM
sequence_length = 180
batch_size = 256
epochs = 50
shuffle_buffer = 10_000
learning_rate = 1e-3
#alpha = 1.5 #How much extra weight do peaks get
#threshold = 1.0 #Target threshold

tf.keras.utils.set_random_seed(16)

validation_start = "2010-10-01"
df = df.sort_values(['gauge_id', 'date']).reset_index(drop=True)

training_df = df[(df['split'] =='train') & (df['date'] < validation_start)]
validation_df = df[(df['split'] =='train') & (df['date'] >= validation_start)]
testing_df = df[df['split']=='test']
print(training_df.shape)
print(validation_df.shape)
print(testing_df.shape)

#Save dataframes for evaluation
testing_df.to_csv("../datacleaning-pipeline/testing_df.csv")
df[(df['split'] =='train')].to_csv("../datacleaning-pipeline/training_df.csv")

#Set up the mean and standard deviation of training data targets
y_mu = float(training_df[target_column].values.reshape(-1,1).mean())
y_sd = float(training_df[target_column].values.reshape(-1,1).std())

def gauge_arrays(gauge):
    #Process data by gauge
    gauge = gauge.sort_values('date')
    x = gauge[input_columns].to_numpy(np.float32)
    y = ((gauge[target_column].to_numpy(np.float32) - y_mu) / y_sd).astype(np.float32)

    #Drop warmup datapoints
    if 'split' in gauge:
        y[(gauge['split'] == 'warmup').to_numpy()] = np.nan
    return x, y
 
 
def make_dataset(df, shuffle):
    #tf.data pipeline yielding batches of ((basin, sequence_length, features), (basin,)).
    datasets, sizes = [], []
    for _, gauge in df.groupby('gauge_id'):
        x, y = gauge_arrays(gauge)
        if len(x) < sequence_length:
            continue
        dataset = keras.utils.timeseries_dataset_from_array(
            x, y[sequence_length - 1:], sequence_length=sequence_length,
            batch_size=None, shuffle=shuffle,
        )
        datasets.append(dataset)
        sizes.append(len(x) - sequence_length + 1)
 
    #Mix gauges randomly so each batch contains many different gauges
    weights = (np.array(sizes) / np.sum(sizes)).tolist()
    dataset = tf.data.Dataset.sample_from_datasets(
        datasets, weights=weights, stop_on_empty_dataset=False
    )
    if shuffle:
        dataset = dataset.shuffle(shuffle_buffer)
    dataset = dataset.map(lambda x, y: (tf.ensure_shape(x, (sequence_length, len(input_columns))), y))#, 1.0 + alpha * tf.nn.relu(y-threshold)))
    return dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
 
 
training_dataset = make_dataset(training_df, shuffle=True)
print(training_dataset)

validation_dataset = make_dataset(validation_df, shuffle=False)
print(validation_dataset)

(119802, 28)
(29930, 28)
(89872, 28)
<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 180, 11), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.float32, name=None))>
<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 180, 11), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.float32, name=None))>


In [5]:
#Define LSTM
model = keras.Sequential(
    [
        layers.Input(shape=(sequence_length, len(input_columns))),
        layers.LSTM(256, return_sequences=True),
        layers.Dropout(0.3),
        layers.LSTM(256),
        layers.Dropout(0.3), 
        layers.Dense(32, activation="relu"),
        layers.Dense(1)
    ]
)

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3)
]

In [6]:
#Compile LSTM
model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=learning_rate),
    loss="mse",
    metrics=[keras.metrics.R2Score(name="r2")]
)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 180, 256)       │       274,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 180, 256)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 256)            │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         8,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 808,001 (3.08 MB)

 Trainable params: 808,001 (3.08 MB)

 Non-trainable params: 0 (0.00 B)

In [7]:
#Evaluate LSTM

keras.config.disable_traceback_filtering()
history = model.fit(
    training_dataset,
    validation_data = validation_dataset,
    epochs = epochs,
    callbacks = callbacks
)

Epoch 1/50


c:\Users\aesas\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


    411/Unknown 300s 720ms/step - loss: 0.1156 - r2: 0.8765

c:\Users\aesas\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\keras\src\trainers\epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


411/411 ━━━━━━━━━━━━━━━━━━━━ 320s 768ms/step - loss: 0.1156 - r2: 0.8765 - val_loss: 0.1630 - val_r2: 0.8352 - learning_rate: 0.0010
Epoch 2/50
411/411 ━━━━━━━━━━━━━━━━━━━━ 516s 1s/step - loss: 0.0686 - r2: 0.9266 - val_loss: 0.1274 - val_r2: 0.8712 - learning_rate: 0.0010
Epoch 3/50
411/411 ━━━━━━━━━━━━━━━━━━━━ 534s 1s/step - loss: 0.0509 - r2: 0.9455 - val_loss: 0.1141 - val_r2: 0.8847 - learning_rate: 0.0010
Epoch 4/50
411/411 ━━━━━━━━━━━━━━━━━━━━ 584s 1s/step - loss: 0.0429 - r2: 0.9541 - val_loss: 0.1145 - val_r2: 0.8843 - learning_rate: 0.0010
Epoch 5/50
411/411 ━━━━━━━━━━━━━━━━━━━━ 636s 2s/step - loss: 0.0358 - r2: 0.9617 - val_loss: 0.1164 - val_r2: 0.8823 - learning_rate: 0.0010
Epoch 6/50
411/411 ━━━━━━━━━━━━━━━━━━━━ 647s 2s/step - loss: 0.0310 - r2: 0.9668 - val_loss: 0.1240 - val_r2: 0.8747 - learning_rate: 0.0010
Epoch 7/50
411/411 ━━━━━━━━━━━━━━━━━━━━ 707s 2s/step - loss: 0.0237 - r2: 0.9747 - val_loss: 0.1041 - val_r2: 0.8948 - learning_rate: 5.0000e-04
Epoch 8/50
411/41

In [11]:
#Save model and training history
model.save("lstm_256x2.keras")
pd.DataFrame(history.history).to_csv("history_256x2.csv", index_label="epoch")

#Current settings:
#sequence_length = 180
#batch_size = 256
#epochs = 50
#shuffle_buffer = 10_000
#learning_rate = 1e-3
#AdamW optimizer
#128 LSTM
#0.3 Dropout
